In [ ]:
from urllib.parse import urlparse

import geopandas as gpd
import rustac
from dask.distributed import Client
from obstore.auth.boto3 import Boto3CredentialProvider
from obstore.store import S3Store
from odc.stac import load
from pystac import ItemCollection

from csdr.utils import load_xarray_stacgeoparquet, xarray_calculate_area

In [ ]:
# url = "https://files.auspatious.com/temp/gmw.parquet"
url = urlparse("s3://files.auspatious.com/temp/gmw.parquet")
bucket = url.netloc
key = url.path.lstrip("/")

store = S3Store(bucket, credential_provider=Boto3CredentialProvider())

bbox = [8.87161, -0.81241, 8.95714, -0.73706]

item_dict = await rustac.read(key, store=store)
items = ItemCollection.from_dict(item_dict)
data = load(items, bbox=bbox).squeeze()
data

In [ ]:
from ipyleaflet import basemaps

masked = data.where(data.mangrove.odc != 0)
masked.mangrove.odc.explore(
    vmin=1, vmax=2, cmap="Greens_r", tiles=basemaps.Esri.WorldImagery
)

In [ ]:
gdf = gpd.read_file(
    "zip+https://files.auspatious.com/testing/unsw/eez/EEZ_land_union_v4_202410.zip!EEZ_land_union_v4_202410/EEZ_land_union_v4_202410.shp"
)

# Get Australia only
gdf_aus = gdf[gdf["UNION"] == "Australia"]
gdf_aus.explore()

In [ ]:
from odc.geo.geom import Geometry

geom = Geometry(gdf_aus.geometry.values[0], crs=gdf.crs)
geom

In [ ]:
data = load_xarray_stacgeoparquet(
    items, geom=geom, chunks={"x": 5000, "y": 5000}, resolution=100, crs="epsg:8857"
).squeeze()
data

In [ ]:
# Use Dask to do the process, as it's potentially big
with Client(processes=4, threads_per_worker=8):
    area = xarray_calculate_area(data, geom, "mangrove", 1)

print(f"Total area in hectares is {area:.2f}")

In [ ]:
total_area = (data.sizes["x"] * data.sizes["y"]) * abs(
    data.odc.geobox.resolution.x * data.odc.geobox.resolution.y
)

percentage_mangrove = (area / total_area) * 100
print(f"Percentage of mangrove cover is {percentage_mangrove:.2f}%")

In [ ]:
from dea_tools.spatial import xr_vectorize

data_gdf = xr_vectorize(data["mangrove"])

# Filter to those with a value of 1.0 and create a union geometry
union = data_gdf[data_gdf.attribute == 1.0].union_all()

# Intersect Australia and the new union geometry
intersection = gdf_aus.to_crs(data_gdf.crs).intersection(union)

# And calculate the area
area_vector = float(intersection.area)

In [ ]:
diff = area - area_vector
print(f"Difference is {diff:.0f}")